# Pengantar: klasifikasi wajah asli vs buatan AI

Notebook ini untuk **mengikuti alurnya langkah demi langkah**. Semua fungsi inti berasal dari paket `faceauth/` di proyek ini.

Notebook ini **tidak menjalankan training**. Training dijalankan dari Terminal dengan `scripts/train.py` (lihat bagian akhir), supaya membuka notebook tidak memicu proses berjam-jam.

Urutan: lingkungan → data dan label → model VGG16 + Flatten → prediksi batch kecil → kebijakan unfreeze → membaca hasil.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import torch
from IPython.display import Image, Markdown, display

from faceauth.config import load_config
from faceauth.data import build_transforms, load_manifest, make_loader
from faceauth.models import build_model, freeze_backbone, summarize, unfreeze_last
from faceauth.utils import CLASS_NAMES, LABEL_MAP, env_info, pick_device

device = pick_device()
print("Device:", device)
env_info()

## 1. Data dan label

Pemetaan label internal: **real = 0, fake = 1**. Kelas positif (untuk ROC/AUC) adalah **fake**.
Manifest dibuat oleh `python scripts/prepare_data.py --download`.

In [ ]:
cfg = load_config("vgg16", "quick")
df, image_root = load_manifest(cfg)
print("Pemetaan label:", LABEL_MAP)
print("image_root:", image_root)
print("\nManifest mode quick (jumlah per split x kelas):")
print(df.pivot_table(index="split", columns="label_name", values="path", aggfunc="count"))
full = pd.read_csv(ROOT / "data/manifests/full.csv")
print("\nDataset lengkap:")
print(full.pivot_table(index="split", columns="label_name", values="path", aggfunc="count"))

In [ ]:
display(Image(filename=str(ROOT / "data/manifests/samples.png")))
display(Markdown((ROOT / "data/manifests/audit.md").read_text()))

## 2. VGG16 dengan Flatten

`VGG16 features (13 conv + 5 max-pool) → Flatten → Linear(256) → ReLU → Dropout(0.5) → Linear(1)`

Output satu **logit**. Probabilitas fake = `sigmoid(logit)`. Tahap A: backbone dibekukan, hanya head yang dilatih.

In [ ]:
spec = build_model(cfg, pretrained=True)
freeze_backbone(spec)
spec.model.to(device)
print(summarize(spec, 224, device))

## 3. Prediksi pada batch kecil gambar asli

Head **belum dilatih**, jadi probabilitas di bawah ini masih acak. Tujuannya hanya membuktikan gambar terbaca, label konsisten, dan model menghasilkan output dengan bentuk yang benar.

In [ ]:
part = df[df["split"] == "train"].groupby("label").head(4)
loader = make_loader(part, image_root, build_transforms(cfg)["eval"], batch_size=8, shuffle=False, num_workers=0)
x, y, idx = next(iter(loader))
spec.model.eval()
with torch.no_grad():
    prob = torch.sigmoid(spec.model(x.to(device))).squeeze(1).cpu()
pd.DataFrame({"file": part["path"].str.split("/").str[-1].values, "label": [CLASS_NAMES[int(v)] for v in y],
              "prob_fake (belum dilatih)": prob.numpy().round(4)})

## 4. Kebijakan unfreeze tahap B

"10 layer terakhir" ditafsirkan sebagai **10 modul terakhir `features`**. ReLU dan MaxPool ikut terhitung walau tidak punya bobot, jadi hanya 4 convolution yang benar-benar dibuka. Ganti di `configs/models/vgg16.yaml` (`policy: last_n_weighted`) bila pembimbing menginginkan 10 layer berbobot.

In [ ]:
report = unfreeze_last(spec, "last_n_modules", 10)
report

## 5. Menjalankan training (dari Terminal, bukan dari sini)

```bash
source .venv/bin/activate
python scripts/train.py --model vgg16 --mode quick     # pemeriksaan cepat (beberapa menit)
python scripts/train.py --model vgg16 --mode subset    # latihan subset 1000+1000
python scripts/compare.py --run-name subset
```

## 6. Membaca hasil (jika sudah ada)

In [ ]:
run = "subset"   # ganti ke "quick" atau "full" sesuai run yang sudah dijalankan
cmp = ROOT / "outputs" / run / "comparison.md"
if cmp.exists():
    display(Markdown(cmp.read_text()))
curves = ROOT / "outputs" / run / "vgg16" / "curves.png"
if curves.exists():
    display(Image(filename=str(curves)))
for st in ("stage_a", "stage_b"):
    roc = ROOT / "outputs" / run / "vgg16" / st / "test" / "roc.png"
    if roc.exists():
        display(Image(filename=str(roc)))
if not cmp.exists():
    print(f"Belum ada hasil untuk run '{run}'. Jalankan training dulu, lalu scripts/compare.py.")